# Global boosting models
LightGBM and XGBoost share the direct multi-horizon Gold feature contract and a Tweedie objective, which is suitable for non-negative, right-skewed demand with many zeros. A small Optuna search is performed on an earlier temporal validation origin using a seeded 10% row sample; the selected parameters are then evaluated on the complete profile.

`item_id`, department, category, store, state, and target event type are stably ordinal-encoded; LightGBM then receives them as native categorical columns instead of treating codes as continuous magnitudes. Missing or unseen categories become `-1`. Numeric fields are coerced to `float32`, remaining nulls become zero, and no standardization is applied because tree splits do not require feature scaling. The target remains in raw units and predictions are clipped at zero. A bounded demand-level factor is learned on the temporal validation origin before horizon-specific residual calibration produces q05/q50/q95.

XGBoost uses CUDA when available; LightGBM uses its portable CPU wheel to keep the local images small. The current search optimizes validation WAPE rather than official WRMSSE, which is a known objective mismatch. SHAP artifacts are generated after final training because fold explanations are expensive and redundant.


In [ ]:
profile = "dev"
run_id = "notebook-boosting"
force = False


In [ ]:
from retail_forecasting.config import load_config
from retail_forecasting.forecasting.workflow import run_single_model
config = load_config(profile)
results = [run_single_model(config, f"{run_id}-{name}", name) for name in ("lightgbm", "xgboost")]
results
